# RAMP model — python-style version
This notebook is reorganized so the assumptions live in a few dictionaries and the model runs from top to bottom without manual box-by-box editing. Leave `USER_CONFIG` and `USER_APPLIANCES` empty to fall back to defaults.

In [ ]:

from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ramp import User, UseCase


In [ ]:

DEFAULT_CONFIG = {
    "input_csv": "residential_input.csv",
    "row_index": 0,
    "households_simulated": 50,
    "num_days": 366,
    "output_csv": "load_profiles_PC1098.csv",
    "save_csv": True,
    "plot_first_day": True,
    "rolling_window": 5,
    "dc_loads_kw": {
        "dc_campus_1_kw": 5000,
        "dc_campus_2_kw": 8000,
        "dc_campus_3_kw": 12000,
    },
    "washing_user_share": 0.30,
}

DEFAULT_APPLIANCES = {
    "Fridge": {
        "enabled": True,
        "user": "households",
        "number": 1,
        "power": 120,
        "func_time": 1440,
        "window_1": [0, 1440],
    },
    "Lighting": {
        "enabled": True,
        "user": "households",
        "number": 5,
        "power": 10,
        "func_time": 120,
        "window_1": [360, 540],
        "window_2": [1080, 1380],
    },
    "Electronics": {
        "enabled": True,
        "user": "households",
        "number": 2,
        "power": 70,
        "func_time": 180,
        "window_1": [500, 900],
        "window_2": [1080, 1320],
    },
    "Cooking": {
        "enabled": True,
        "user": "households",
        "number": 1,
        "power": 800,
        "func_time": 60,
        "window_1": [1020, 1380],
    },
    "WashingMachine": {
        "enabled": True,
        "user": "washing_users",
        "number": 1,
        "power": 500,
        "func_time": 60,
        "window_1": [600, 1200],
    },
    "BaseLoad": {
        "enabled": True,
        "user": "households",
        "number": 1,
        "power": 150,
        "func_time": 1440,
        "window_1": [0, 1440],
    },
}


In [ ]:

# Optional scenario-specific changes.
# Leave empty to use all defaults.
USER_CONFIG = {
    # "households_simulated": 100,
    # "num_days": 365,
}

USER_APPLIANCES = {
    # Existing appliance override example:
    # "Cooking": {"power": 1000},

    # New appliance example:
    # "Laptop": {
    #     "enabled": True,
    #     "user": "households",
    #     "number": 1,
    #     "power": 60,
    #     "func_time": 240,
    #     "window_1": [1080, 1380],
    # },
}


In [ ]:

def deep_merge(default, override):
    result = deepcopy(default)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    return result

config = deep_merge(DEFAULT_CONFIG, USER_CONFIG)
appliances = deep_merge(DEFAULT_APPLIANCES, USER_APPLIANCES)

config, list(appliances.keys())[:5]


In [ ]:

REQUIRED_INPUT_COLUMNS = ["households", "electricity_per_household_kwh"]

def load_residential_inputs(csv_path, row_index=0):
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)
    missing = [col for col in REQUIRED_INPUT_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"CSV missing required columns: {missing}. Found: {list(df.columns)}")
    row = df.loc[row_index]
    return {
        "df": df,
        "households_real": int(row["households"]),
        "electricity_per_household_kwh": float(row["electricity_per_household_kwh"]),
    }

def build_users(config):
    households_simulated = int(config["households_simulated"])
    washing_users = int(round(config["washing_user_share"] * households_simulated))
    washing_users = max(washing_users, 1) if config["washing_user_share"] > 0 else 0

    users = {"households": User("households", households_simulated)}
    if washing_users > 0:
        users["washing_users"] = User("washing_users", washing_users)
    return users

ALLOWED_APPLIANCE_KEYS = {
    "name", "number", "power", "func_time",
    "window_1", "window_2", "window_3",
    "fixed", "fixed_cycle", "occasional_use", "flat",
    "thermal_p_var", "pref_index", "wd_we_type",
    "func_cycle", "time_fraction_random_variability",
    "num_windows", "continuous_duty_cycle",
}

def add_appliances(users, appliances):
    for appliance_name, params in appliances.items():
        if not params.get("enabled", True):
            continue
        user_key = params["user"]
        if user_key not in users:
            raise KeyError(f"Unknown user group: {user_key}. Available: {list(users)}")

        ramp_kwargs = {"name": appliance_name}
        for key, value in params.items():
            if key in {"enabled", "user"}:
                continue
            if key in ALLOWED_APPLIANCE_KEYS:
                ramp_kwargs[key] = value
        users[user_key].add_appliance(**ramp_kwargs)


In [ ]:

input_data = load_residential_inputs(config["input_csv"], config["row_index"])
households_real = input_data["households_real"]
electricity_per_household_kwh = input_data["electricity_per_household_kwh"]

users = build_users(config)
add_appliances(users, appliances)

case = UseCase(users=list(users.values()))
case.initialize(num_days=int(config["num_days"]))

profiles = case.generate_daily_load_profiles(flat=True, verbose=False)
profile = np.array(profiles)

weight = households_real / config["households_simulated"]
profile_real = profile * weight
profile_kw = profile_real / 1000.0

annual_energy = profile_kw.sum() / 60
daily_energy = annual_energy / 365

print("Households:", households_real)
print("Electricity per household:", electricity_per_household_kwh)
print("Scaling factor:", weight)
print("Peak load (kW):", profile_kw.max())
print("Annual energy (kWh):", annual_energy)
print("Average daily energy (kWh):", daily_energy)


In [ ]:

output = pd.DataFrame({
    "residential_kw": profile_kw,
    **{name: np.ones_like(profile_kw) * load for name, load in config["dc_loads_kw"].items()}
})

if config["save_csv"]:
    output.to_csv(config["output_csv"], index=False)
    print(f"Saved to {config['output_csv']}")

output.head()


In [ ]:

if config["plot_first_day"]:
    smoothed = pd.Series(profile_kw).rolling(config["rolling_window"], min_periods=1).mean()
    plt.figure(figsize=(10, 4))
    plt.plot(smoothed[:1440])
    plt.xlabel("Minute of day")
    plt.ylabel("Load (kW)")
    plt.title("Residential electricity demand")
    plt.show()
